# Car Brand Classification — training in Google Colab

Trains all three complexity levels and builds the comparison:

| Level | Model | Runs on | Approx. time |
|---|---|---|---|
| 1 | Logistic Regression + Random Forest (5-fold CV) | CPU | ~15–25 min |
| 2 | ResNet18 (fine-tuned) | GPU | ~10–15 min |
| 3 | ViT-Small/16 (fine-tuned) | GPU | ~15–25 min |

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

Results (figures, tables, metrics) are copied to Google Drive after each step
(`MyDrive/car-brand-classification/`), so nothing is lost if the session disconnects,
and downloaded as `results.zip` at the end.

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
REPO = "https://github.com/VolodymyrDem/car-brand-classification.git"
WORKDIR = "/content/car-brand-classification"
if not os.path.exists(WORKDIR):
    !git clone -q {REPO} {WORKDIR}
%cd {WORKDIR}
!git pull -q
!pip install -q timm tabulate
!git log --oneline -1

In [ ]:
# Backup of results to Google Drive (set USE_DRIVE = False to skip)
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/car-brand-classification"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

def backup():
    if USE_DRIVE:
        !rsync -a reports/ {DRIVE_DIR}/reports/
        !rsync -a --include='*.json' --include='*.pt' --exclude='*' models/ {DRIVE_DIR}/models/
        print("Backed up to", DRIVE_DIR)

## 1. Data: download Stanford Cars, group models into makes, stratified 70/15/15 split

In [ ]:
!python scripts/prepare_data.py
!python scripts/eda.py

## 2. Level 1 — classical ML (Logistic Regression, Random Forest), 5-fold cross-validation

In [ ]:
!python scripts/train_classical.py
backup()

## 3. Level 2 — ResNet18 (CNN, transfer learning)

In [ ]:
!python scripts/train_deep.py --model resnet18 --workers 2
backup()

## 4. Level 3 — ViT-Small/16 (transformer, transfer learning)

In [ ]:
!python scripts/train_deep.py --model vit_small --workers 2
backup()

## 5. Comparison

In [ ]:
!python scripts/compare.py
backup()

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for name in ["comparison_metrics", "comparison_quality_vs_time",
             "learning_curves_resnet18", "learning_curves_vit_small",
             "confusion_matrix_logreg", "confusion_matrix_random_forest",
             "confusion_matrix_resnet18", "confusion_matrix_vit_small"]:
    p = Path("reports/figures") / f"{name}.png"
    if p.exists():
        print(name); display(Image(str(p), width=900))

## 6. Download results

In [ ]:
!zip -qr results.zip reports
from google.colab import files
files.download("results.zip")